# Story figures

The presentation's figures, drawn from the tables in `outputs/tables` (written by the analysis and comparison notebooks). Each figure is saved to `outputs/figures`.

In [ ]:
# Story figures for the presentation: read the tables saved by the analysis notebook, write PNGs to outputs/figures
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

TABLES, FIGURES = "../outputs/tables", "../outputs/figures"
os.makedirs(FIGURES, exist_ok=True)
CITIES = {"London": "#2a78d6", "Paris": "#eb6834"}
INK, INK_2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#a3a29c", "#e6e5e1", "#fcfcfb"
GOOD, CRITICAL, WARNING = "#0ca30c", "#d03b3b", "#b57f00"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.size": 11, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.edgecolor": MUTED, "axes.labelcolor": INK_2, "xtick.color": INK_2, "ytick.color": INK_2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.8, "axes.axisbelow": True, "legend.frameon": False,
})

LABELS = {
    "superhost": "Superhost", "host_years_z": "Host tenure (+1 SD)", "log_host_listings_z": "Host's listings (+1 SD)",
    "accommodates_z": "Capacity (+1 SD)", "n_amenities_z": "Amenities (+1 SD)", "entire_home": "Entire home",
    "log_price_z": "Price (+1 SD)", "price_missing": "No price (not bookable)",
    "has_profile_pic": "Host profile photo", "identity_verified": "Identity verified", "log_desc_words_z": "Description length",
    "lix_z": "Description readability (LIX)", "promo_per_100_z": "Promotional words", "desc_pos_z": "Description positivity",
    "desc_neg_z": "Description negativity", "review_scores_rating_z": "Overall rating",
    "review_scores_communication_z": "Communication score", "review_scores_cleanliness_z": "Cleanliness score",
    "share_positive_z": "Share of positive reviews", "review_neg_z": "Review negativity",
    "host_response_time": "Host response time", "instant_bookable": "Instant booking",
}


def table(name, city=None):
    return pd.read_csv(f"{TABLES}/{city + '_' if city else ''}{name}.csv", index_col=0)


def title(fig, headline, sub, top_in=1.25):
    h = fig.get_figheight()
    fig.text(0.01, 1 - 0.3 / h, headline, ha="left", va="center", fontsize=15, fontweight="bold", color=INK)
    fig.text(0.01, 1 - 0.65 / h, sub, ha="left", va="center", fontsize=11, color=INK_2)
    fig.subplots_adjust(top=1 - top_in / h)


def save(fig, name):
    fig.savefig(f"{FIGURES}/{name}.png", dpi=200, bbox_inches="tight")
    plt.show()


def forest(ax, tables, label, xlabel, order=None, ticks=(0.25, 0.5, 0.75, 1, 1.5, 2, 3, 4)):
    """IRR or odds-ratio dot plot with 95% CI, one row per predictor, cities offset vertically."""
    order = tables["London"].sort_values(label).index if order is None else order
    y = np.arange(len(order))
    for k, (city, t) in enumerate(tables.items()):
        t = t.reindex(order)
        off = 0.18 if k == 0 else -0.18
        ax.hlines(y + off, t["low"], t["high"], color=CITIES[city], linewidth=2)
        ax.plot(t[label], y + off, "o", ms=8, color=CITIES[city], markeredgecolor=SURFACE, markeredgewidth=2, label=city)
    ax.axvline(1, color=INK_2, linewidth=1)
    ax.set_xscale("log")
    ax.set_xticks(ticks, [f"{t:g}×" for t in ticks])
    ax.set_xlim(ticks[0] * 0.85, ticks[-1] * 1.15)
    ax.minorticks_off()
    ax.set_yticks(y, [LABELS.get(v, v) for v in order])
    ax.grid(axis="y", visible=False)
    ax.set_xlabel(xlabel)

In [ ]:
# 1. The problem: many listings are never reviewed
fit = {c: table("count_fit", c) for c in CITIES}
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(fit["London"]))
for k, (city, t) in enumerate(fit.items()):
    bars = ax.bar(x + (k - 0.5) * 0.4, 100 * t["Observed"], width=0.38, color=CITIES[city], label=city)
    ax.bar_label(bars, [f"{100 * t['Observed'].iloc[0]:.0f}%"] + [""] * (len(t) - 1), padding=3, color=INK, fontweight="bold")
ax.set_xticks(x, fit["London"].index)
ax.set_xlabel("Number of reviews")
ax.set_ylabel("% of listings")
ax.legend(loc="upper right")
title(fig, "About 1 in 5 listings has never been reviewed",
      "Share of listings by lifetime number of reviews (proxy for bookings), 50,000 sampled listings per city")
save(fig, "01_never_reviewed")

In [ ]:
# 2. Why Negative Binomial: it reproduces the zeros and the long tail; Poisson does not
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), sharey=True)
for ax, (city, t) in zip(axes, fit.items()):
    x = np.arange(len(t))
    ax.bar(x, 100 * t["Observed"], width=0.7, color=CITIES[city], alpha=0.35, label="Observed")
    ax.plot(x, 100 * t["Negative Binomial"], "o-", color=INK, linewidth=2, ms=8, label="Negative Binomial")
    ax.plot(x, 100 * t["Poisson"], "s--", color=MUTED, linewidth=2, ms=7, label="Poisson")
    m = table("model_fit", city)
    ax.set_title(f"{city}  ·  AIC: NB {m.loc['AIC', 'Negative Binomial'] / 1e3:,.0f}k vs Poisson {m.loc['AIC', 'Poisson'] / 1e3:,.0f}k",
                 color=INK)
    ax.set_xticks(x, t.index, fontsize=9)
    ax.set_xlabel("Number of reviews")
axes[0].set_ylabel("% of listings")
axes[0].legend(loc="upper right")
title(fig, "Poisson predicts no zero-review listings; Negative Binomial matches the data",
      "Observed vs predicted distribution of reviews. Variance is 125× (London) and 138× (Paris) the mean, where Poisson assumes 1×")
save(fig, "02_why_negative_binomial")

In [ ]:
# 3. What drives bookings (Model A)
fig, ax = plt.subplots(figsize=(10, 6))
forest(ax, {c: table("model_a", c) for c in CITIES}, "IRR", "Multiplier on expected reviews (IRR, log scale, 95% CI)")
ax.legend(loc="lower right")
title(fig, "Superhost status and amenities matter most in both cities",
      "Negative Binomial, all listings; standard errors clustered by host. Right of 1× = more reviews")
save(fig, "03_drivers")

In [ ]:
# 4. First booking vs more bookings (hurdle model)
fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharey=True)
forest(axes[0], {c: table("hurdle_any_review", c) for c in CITIES}, "IRR", "Odds ratio (log scale, 95% CI)", ticks=(0.5, 0.75, 1, 1.5, 2, 3, 4))
axes[0].set_title("Getting a first review", color=INK)
forest(axes[1], {c: table("hurdle_how_many", c) for c in CITIES}, "IRR",
       "Multiplier on reviews, reviewed listings (IRR, log scale)", order=table("hurdle_any_review", "London").sort_values("IRR").index, ticks=(0.5, 0.75, 1, 1.5, 2, 3, 4))
axes[1].set_title("Getting more reviews once reviewed", color=INK)
axes[1].legend(loc="lower right")
title(fig, "Superhost status and amenities help at both stages; entire homes lose out only on repeat reviews",
      "Hurdle model: logit for any review (left), zero-truncated Negative Binomial for how many among reviewed listings (right)")
fig.subplots_adjust(wspace=0.14)
save(fig, "04_first_vs_more")

In [ ]:
# 5. What a host can change
levers = {"Everybody is a superhost": "Become a superhost", "10 more amenities": "Add 10 amenities", "Price 10% lower": "Cut price by 10%"}
fig, ax = plt.subplots(figsize=(10, 4.6))
y = np.arange(len(levers))
for k, city in enumerate(CITIES):
    t = table("scenarios", city)
    change = 100 * (t.loc[list(levers), "Expected reviews"] / t.loc["As observed", "Expected reviews"] - 1)
    bars = ax.barh(y + (-0.2 if k == 0 else 0.2), change, height=0.38, color=CITIES[city], label=city)
    ax.bar_label(bars, [f"+{v:.0f}%" for v in change], padding=4, color=INK)
ax.set_yticks(y, list(levers.values()))
ax.invert_yaxis()
ax.grid(axis="y", visible=False)
ax.set_xlabel("Change in expected reviews per listing vs today (%)")
ax.legend(loc="lower right")
title(fig, "Becoming a superhost is the biggest lever a host controls",
      "Model A predictions, averaged over all listings with one feature changed and the rest as observed")
save(fig, "05_host_levers")

In [ ]:
# 6. Tenure: lifetime reviews vs reviews in the last 12 months
fig, axes = plt.subplots(1, 2, figsize=(13, 5.6), sharey=True)
order = table("model_a", "London").sort_values("IRR").index
for ax, city in zip(axes, CITIES):
    life, recent = table("model_a", city).reindex(order), table("model_ltm", city).reindex(order)
    y = np.arange(len(order))
    ax.hlines(y, life["IRR"], recent["IRR"], color=MUTED, linewidth=2)
    ax.plot(life["IRR"], y, "o", ms=9, markerfacecolor=SURFACE, markeredgecolor=CITIES[city], markeredgewidth=2, label="Lifetime reviews")
    ax.plot(recent["IRR"], y, "o", ms=9, color=CITIES[city], markeredgecolor=SURFACE, markeredgewidth=2, label="Reviews, last 12 months")
    ax.axvline(1, color=INK_2, linewidth=1)
    ax.set_xscale("log")
    ax.set_xticks([0.25, 0.5, 1, 2, 3], ["0.25×", "0.5×", "1×", "2×", "3×"])
    ax.minorticks_off()
    ax.grid(axis="y", visible=False)
    ax.set_title(city, color=INK)
    ax.set_xlabel("IRR (log scale)")
    ax.legend(loc="upper left")
axes[0].set_yticks(y, [LABELS[v] for v in order])
title(fig, "Tenure's effect comes from old listings piling up reviews, not from active hosting",
      "Model A with lifetime reviews (hollow) vs reviews in the last 12 months (filled)")
fig.subplots_adjust(wspace=0.08)
save(fig, "06_tenure_lifetime_vs_recent")

In [ ]:
# 7. Replication scorecard
card = table("scorecard")
fig, ax = plt.subplots(figsize=(10, 0.42 * len(card) + 1.6))
ax.axis("off")
ax.set_xlim(0, 11.5)
ax.set_ylim(len(card) + 0.5, -1)
for col, x in [("Finding in the literature", 0), ("Source", 5.0), ("London", 6.7), ("Paris", 9.1)]:
    ax.text(x, -0.4, col, fontweight="bold", color=INK, va="center")
for i, (var, r) in enumerate(card.iterrows()):
    direction = {"+": "more reviews", "-": "fewer reviews", "faster +": "more reviews if faster"}[r["expected"]]
    ax.text(0, i + 0.5, f"{LABELS.get(var, var)} → {direction}", color=INK, va="center")
    ax.text(5.0, i + 0.5, r["source"], color=INK_2, va="center")
    for c, x in [("London", 6.7), ("Paris", 9.1)]:
        v = r[c]
        if v.startswith("✅"):
            sym, color, txt = "✓", GOOD, f"Replicated {v.split()[-1]}×"
        elif v.startswith("❌"):
            sym, color, txt = "✗", CRITICAL, f"Opposite {v.split()[-1]}×"
        elif v.startswith("⚠️"):
            sym, color, txt = "!", WARNING, "Not in data"
        else:
            sym, color, txt = "–", MUTED, "Not significant"
        ax.text(x, i + 0.5, sym, color=color, fontweight="bold", fontsize=13, va="center")
        ax.text(x + 0.3, i + 0.5, txt, color=INK_2, va="center")
    ax.axhline(i + 1, color=GRID, linewidth=0.8)
n = {c: card[c].str.startswith("✅").sum() for c in CITIES}
title(fig, f"Replicated: {n['London']} of 18 findings in London, {n['Paris']} of 18 in Paris",
      "Biswas et al. (2020), Sengupta et al. (2021) vs our Models A, A+ and B. Two predictors cannot be tested: the fields are empty")
fig.subplots_adjust(left=0.02, right=0.98)
save(fig, "07_replication_scorecard")